# Mass-Editing Memory in a Transformer
This notebook enables interactive experimentation with MEMIT and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution

Here, you can specify a GPT model (`MODEL_NAME`).

* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM


In [2]:
MODEL_NAME = "EleutherAI/gpt-j-6B"
# MODEL_NAME = "gpt2-xl"

In [ ]:
CACHE_DIR = None # Optional cache directory for the model

model, tok = (
    AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        cache_dir=CACHE_DIR,
        low_cpu_mem_usage=False,
        torch_dtype=(torch.float16 if "20b" in MODEL_NAME else None),
    ).to("cuda"),
    AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR),
)
tok.pad_token = tok.eos_token
model.config

A requested rewrite can be specified using `request`. `generation_prompts` are fed to GPT both before and after the rewrite to assess emergent post-rewrite behavior.


In [15]:
def single_request(s, r, o):
    """
    Creates a single request from 
        subject
        relation where "{}" will be replaced by the subject field
        object
    Returns created request and single generation prompt where relation is injected with subject
    """
    request = [{
        "prompt": r,
        "subject": s,
        "target_new": {"str": o},
    }]
    
    generation_prompt = [
        r.format(s)
    ]
    return (request, generation_prompt)

This cell executes the model edit.
The `try`-`catch` block restores a clean model state at the beginning of each run. `ALG_NAME` controls which algorithm is used. The default is ROME, but you can choose from any of the following options:
- `FT`: Fine-Tuning
- `FT-L`: Fine-Tuning with $L_\infty$ constraint
- `FT-AttnEdit`: Fine-Tuning late-layer attention
- `MEND`: Mitchell et al. Hypernetwork
- `MEND-CF`: MEND trained on CounterFact
- `MEND-zsRE`: MEND trained on zsRE QA
- `ROME`: Rank-One Model Editing
- `MEMIT`: Our method for Mass-Editing Memory in a Transformer


Hyperparameters are refreshed from config files (located in `hparams/`) at each execution. To modify any parameter, edit and save the respective file. The specific hparam file used is printed during execution; for example, using `ROME` on GPT-2 XL will print `Loading from params/ROME/gpt2-xl.json`.

ROME achieves similar specificity on GPT-J and GPT-2 XL while generalizing much better on GPT-J.


In [5]:
ALG_NAME = "ROME"

In [14]:
def restore():
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("Original model restored")
    except NameError as e:
        print(f"No model weights to restore: {e}")

def rewrite():
    model_new, orig_weights = demo_model_editing(
        model, tok, request, generation_prompts, alg_name=ALG_NAME
    )

def restore_rewrite():
    restore_model()
    rewrite_model()

In [45]:
# How much new tokens we want to generate for each inference run
def get_tok_cnt(prompt, new_tokens = 50):
    prompt_tok_size = tok(prompt, padding=True, return_tensors="pt")["input_ids"].shape[1]
    return prompt_tok_size + new_tokens

# Prints out only newly generated parts (discards the original prompt)
def print_new(output, prompt):
    p = prompt[0] # prompt is in form of a list but contains only single element
    for o in output:
        print(o[len(p):])

# Usage of KE in code editing

The goal of the following section is to experiment with usage of KE methods in generating code. We will try to influence the model understanding of programming concepts and see, whether it is enough to trigger changes when generating snippets of code.

### Experiment 1

Trying to change representation of keywords in python language. Let's start with representation of keyword `return`. First, I wanted to generate such prompt, to check whether the LM has the knowledge that `return` represents end of function. I keep the following cell to show a **badly** created prompt. As we can see, the outputs of the model are not consistent.  

In [ ]:
s = "Python return statement"
r = "{} is represented by keyword"
o = "exit"
req, gen_prompt = single_request(s, r, o)

# First generate from unedited model, 20 samples of 3 tokens
output = generate_fast(model, tok, gen_prompt, n_gen_per_prompt=20, max_out_len=get_tok_cnt(gen_prompt, 3))

# Printing only the new tokens
print_new(output, gen_prompt)

In [ ]:
s = "keyword used to exit a function"
r = "In Python, the {} and give a value back is"
o = "exit"
req, gen_prompt = single_request(s, r, o)

# First generate from unedited model, 20 samples of 3 tokens
output = generate_fast(model, tok, gen_prompt, n_gen_per_prompt=20, max_out_len=get_tok_cnt(gen_prompt, 3))

# Printing only the new tokens
print_new(output, gen_prompt)